In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


DATA_PATH = "Nitrate_Benchmark_2.csv" 

RAND_SEED = 42

KINETICS_MODE = "dual_linear"   

INPUT_COLUMNS_STAGE3 = [
    'dist_x', 'time', 'C_DOC', 'porosity', 'velocity', 'dispersivity',
    'C_NO3', 'k_Fe', 'k_C', 'C_Fe3'  
]
INPUT_COLUMNS_STAGE5 = [
    'dist_x', 'time', 'C_DOC', 'porosity', 'velocity', 'dispersivity',
    'C_NO3', 'k_Fe', 'k_C', 'C_Fe3', 'K_Fe'
]

output_column = 'N(5)(mol/kgw)'

CANDIDATE_CONST_COLS = [
    'porosity', 'velocity', 'dispersivity', 'k_Fe', 'k_C', 'C_Fe3', 'K_Fe'
]

BATCH_SIZE     = 65536
EPOCHS_ADAM    = 2000
LR_ADAM        = 1e-3
LBFGS_MAX_ITER = 2000
WD_ADAMW       = 1e-4

USE_COLLOCATION   = True
COLLOC_MULTIPLIER = 0.5

lambda_phys_base  = 0.0
lambda_phys_max   = 1.0
lambda_redox_base = 0.0
lambda_redox_max  = 0.2
lambda_nonneg     = 0.1
warmup_epochs     = 400

lambda_bc_in  = 1.0
lambda_bc_out = 1.0


df = pd.read_csv(DATA_PATH)
df = df[df.iloc[:, 5] != -99].copy()

CANDIDATE_CONST_COLS = [c for c in CANDIDATE_CONST_COLS if c in df.columns]
if len(CANDIDATE_CONST_COLS) == 0:
    raise ValueError("No invariant candidate columns found")


if KINETICS_MODE == "dual_linear":
    input_columns = [c for c in INPUT_COLUMNS_STAGE3 if c in df.columns]
    if 'FeOH3_mass' in df.columns and 'FeOH3_mass' not in input_columns:
        input_columns.append('FeOH3_mass') 
elif KINETICS_MODE == "dual_monod":
    input_columns = [c for c in INPUT_COLUMNS_STAGE5 if c in df.columns]
else:
    raise ValueError(f"Unknown KINETICS_MODE={KINETICS_MODE}")

missing = [c for c in input_columns if c not in df.columns]
if missing:
    raise ValueError(f"Required input columns missing in CSV: {missing}")


const_block = df[CANDIDATE_CONST_COLS].copy()
for c in CANDIDATE_CONST_COLS:
    if pd.api.types.is_numeric_dtype(const_block[c]):
        const_block[c] = const_block[c].round(8)  
sim_key = const_block.astype(str).agg('|'.join, axis=1)
sim_id, _ = pd.factorize(sim_key, sort=True)
df['__sim_id__'] = sim_id

n_sims = df['__sim_id__'].nunique()
print(f"Detected unique simulations (via invariant-parameter signature): {n_sims}")


rng = np.random.default_rng(RAND_SEED)
all_sims = np.array(sorted(df['__sim_id__'].unique()))
rng.shuffle(all_sims)

if n_sims >= 200:
    n_train_sims, n_test_sims = 160, 40
elif n_sims >= 2:
    n_test_sims  = max(1, int(round(0.2 * n_sims)))
    n_train_sims = n_sims - n_test_sims
    print(f"Warning: expected 200 sims, found {n_sims}. Using {n_train_sims} train / {n_test_sims} test.")
else:
    raise ValueError("Less than 2 simulation groups detected; cannot split.")

train_sims = set(all_sims[:n_train_sims])
test_sims  = set(all_sims[n_train_sims:n_train_sims + n_test_sims])

df_train = df[df['__sim_id__'].isin(train_sims)].copy()
df_test  = df[df['__sim_id__'].isin(test_sims)].copy()

print(f"Train simulations: {len(train_sims)} | Train rows: {len(df_train)}")
print(f"Test  simulations: {len(test_sims)} | Test  rows: {len(df_test)}")


x_min_per_sim_all = df.groupby('__sim_id__')['dist_x'].transform('min')
is_inlet_all = (df['dist_x'].values == x_min_per_sim_all.values)
ref_map = (
    df.loc[is_inlet_all, ['__sim_id__', 'time', 'C_NO3']]
      .rename(columns={'C_NO3': 'Cref'})
      .copy()
)

df_train = df_train.merge(ref_map, on=['__sim_id__', 'time'], how='left')
df_test  = df_test.merge(ref_map,  on=['__sim_id__', 'time'], how='left')


X_train_raw = df_train[input_columns].to_numpy(dtype=float)
y_train_raw = df_train[output_column].to_numpy(dtype=float).reshape(-1, 1)

X_test_raw  = df_test[input_columns].to_numpy(dtype=float)
y_test_raw  = df_test[output_column].to_numpy(dtype=float).reshape(-1, 1)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(X_train_raw)
y_train = scaler_y.fit_transform(y_train_raw)

X_test  = scaler_X.transform(X_test_raw)
y_test  = scaler_y.transform(y_test_raw)

Cref_train = df_train['Cref'].to_numpy(dtype=float).reshape(-1, 1)
Cref_test  = df_test['Cref'].to_numpy(dtype=float).reshape(-1, 1)


ix_x       = input_columns.index('dist_x')
ix_t       = input_columns.index('time')
ix_DOC     = input_columns.index('C_DOC')
ix_phi     = input_columns.index('porosity')
ix_v       = input_columns.index('velocity')
ix_alphaL  = input_columns.index('dispersivity')
ix_kFe     = input_columns.index('k_Fe')       if 'k_Fe'       in input_columns else None
ix_kC      = input_columns.index('k_C')        if 'k_C'        in input_columns else None
ix_Fe3     = input_columns.index('C_Fe3')      if 'C_Fe3'      in input_columns else None
ix_KFe     = input_columns.index('K_Fe')       if 'K_Fe'       in input_columns else None
ix_FeOH3   = input_columns.index('FeOH3_mass') if 'FeOH3_mass' in input_columns else None

def _to_torch_vec(arr, device):
    return torch.tensor(arr, dtype=torch.float32, device=device)


class SurrogatePINN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 64),  nn.Tanh(),
            nn.Linear(64, 64),   nn.Tanh(),
            nn.Linear(64, 32),   nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


def compute_nonnegativity_penalty(C_phys):
    c = np.asarray(C_phys).reshape(-1)
    neg = c < 0
    return float(np.mean(c[neg]**2)) if np.any(neg) else 0.0

def compute_redox_sequence_penalty(df_phys, window=3, pe_min=-300.0, pe_max=0.0,
                                   eps_NO3=1e-6, eps_Fe3=1e-9):
    if 'N(5)(mol/kgw)_predicted' not in df_phys.columns or 'C_Fe3' not in df_phys.columns:
        return 0.0

    Df = df_phys.sort_values(['dist_x', 'time']).reset_index(drop=True)
    pe_v = Df['pe'].values.astype(float) if 'pe' in Df.columns else np.zeros(len(Df))
    NO3p = Df['N(5)(mol/kgw)_predicted'].values.astype(float)
    Fe3  = Df['C_Fe3'].values.astype(float)

    NO3_s = pd.Series(NO3p).rolling(window=window, min_periods=1, center=True).mean().values
    Fe3_s = pd.Series(Fe3).rolling(window=window, min_periods=1, center=True).mean().values
    dFe3  = np.diff(Fe3_s, prepend=Fe3_s[0])

    pe_w = np.zeros_like(pe_v)
    pe_w[pe_v <= pe_min] = 1.0
    mid = (pe_v > pe_min) & (pe_v < pe_max)
    if np.any(mid):
        pe_w[mid] = (pe_max - pe_v[mid]) / (pe_max - pe_min)

    mask = (NO3_s > eps_NO3) & (Fe3_s > eps_Fe3) & (dFe3 < 0)
    if not np.any(mask):
        return 0.0

    Fe3_mean = max(np.mean(Fe3_s), eps_Fe3)
    rel_drop = np.minimum((-dFe3[mask]) / Fe3_mean, 2.0)
    w = pe_w[mask]
    if rel_drop.size == 0:
        return 0.0
    return float(np.mean(rel_drop * w))

def pde_residual_mean_abs(C_scaled, xb_scaled, mu_X_t, std_X_t, mu_y_t, std_y_t, device):
    if not xb_scaled.requires_grad:
        xb_scaled.requires_grad_(True)

    C = C_scaled * std_y_t + mu_y_t  

    std_x = std_X_t[ix_x].view(1, 1)
    std_t = std_X_t[ix_t].view(1, 1)

    ones = torch.ones_like(C)
    grad_C = torch.autograd.grad(C, xb_scaled, grad_outputs=ones,
                                 create_graph=True, retain_graph=True)[0]
    dCdx_s = grad_C[:, ix_x:ix_x+1]
    dCdt_s = grad_C[:, ix_t:ix_t+1]

    dCdx = dCdx_s / std_x
    dCdt = dCdt_s / std_t

    grad_dCdx_s = torch.autograd.grad(dCdx_s, xb_scaled,
                                      grad_outputs=torch.ones_like(dCdx_s),
                                      create_graph=True, retain_graph=True)[0]
    d2Cdx2_s = grad_dCdx_s[:, ix_x:ix_x+1]
    d2Cdx2   = d2Cdx2_s / (std_x * std_x)

    xb_phys = xb_scaled * std_X_t + mu_X_t
    phi     = xb_phys[:, ix_phi:ix_phi+1]
    v       = xb_phys[:, ix_v:ix_v+1]
    alphaL  = xb_phys[:, ix_alphaL:ix_alphaL+1]
    DL      = alphaL * v

    C_NO3 = C.clamp_min(0.0)                              
    C_DOC = xb_phys[:, ix_DOC:ix_DOC+1].clamp_min(0.0)     

    if KINETICS_MODE == "dual_linear":
        kC   = xb_phys[:, ix_kC:ix_kC+1]   if ix_kC   is not None else torch.zeros_like(C)
        kFe  = xb_phys[:, ix_kFe:ix_kFe+1] if ix_kFe  is not None else torch.zeros_like(C)
        FeOH3 = (xb_phys[:, ix_FeOH3:ix_FeOH3+1].clamp_min(0.0)
                 if ix_FeOH3 is not None else torch.zeros_like(C))

        r_NO3 = kC  * C_NO3 * C_DOC
        r_Fe  = kFe * FeOH3 * C_DOC
        R     = r_NO3 + r_Fe

    elif KINETICS_MODE == "dual_monod":
        kC   = xb_phys[:, ix_kC:ix_kC+1]   if ix_kC   is not None else torch.zeros_like(C)
        kFe  = xb_phys[:, ix_kFe:ix_kFe+1] if ix_kFe  is not None else torch.zeros_like(C)
        KFe  = xb_phys[:, ix_KFe:ix_KFe+1] if ix_KFe  is not None else torch.full_like(C, 1.0)
        FeM  = xb_phys[:, ix_Fe3:ix_Fe3+1].clamp_min(0.0)  if ix_Fe3 is not None else torch.zeros_like(C)

        eps  = 1e-12
        KNO3 = torch.full_like(C_NO3, 1e-6)
        KDOC = torch.full_like(C_DOC, 1e-6)

        r_NO3 = kC  * (C_NO3 / (KNO3 + C_NO3 + eps)) * (C_DOC / (KDOC + C_DOC + eps))
        r_Fe  = kFe * (FeM  / (KFe  + FeM  + eps))   * (C_DOC / (KDOC + C_DOC + eps))
        R     = r_NO3 + r_Fe
    else:
        raise ValueError(f"Unknown KINETICS_MODE={KINETICS_MODE}")

    residual = phi * dCdt + v * dCdx - (phi * DL) * d2Cdx2 - R
    return residual.abs().mean()

def bc_losses(model, X_inlet_scaled, X_outlet_scaled, Cref_inlet_phys, mu_X_t, std_X_t, mu_y_t, std_y_t):
    pred_in_scaled = model(X_inlet_scaled)
    C_in_phys = pred_in_scaled * std_y_t + mu_y_t
    loss_inlet = torch.mean((C_in_phys - Cref_inlet_phys) ** 2)

    pred_out_scaled = model(X_outlet_scaled)
    C_out_phys = pred_out_scaled * std_y_t + mu_y_t

    ones = torch.ones_like(C_out_phys)
    grad_C = torch.autograd.grad(C_out_phys, X_outlet_scaled, grad_outputs=ones,
                                 create_graph=True, retain_graph=True)[0]
    dCdx_s = grad_C[:, ix_x:ix_x+1]
    std_x  = std_X_t[ix_x].view(1,1)
    dCdx_phys = dCdx_s / std_x

    loss_outlet = torch.mean(dCdx_phys ** 2)  
    return loss_inlet, loss_outlet


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RAND_SEED)
np.random.seed(RAND_SEED)

model = SurrogatePINN(input_dim=len(input_columns)).to(device)

X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)

mu_X_t  = torch.tensor(scaler_X.mean_,  dtype=torch.float32, device=device)   
std_X_t = torch.tensor(scaler_X.scale_, dtype=torch.float32, device=device)  
mu_y_t  = torch.tensor(scaler_y.mean_,  dtype=torch.float32, device=device)   
std_y_t = torch.tensor(scaler_y.scale_, dtype=torch.float32, device=device)   

Cref_train_t = torch.tensor(Cref_train, dtype=torch.float32, device=device)  
Cref_test_t  = torch.tensor(Cref_test,  dtype=torch.float32, device=device)  

optimizer = optim.AdamW(model.parameters(), lr=LR_ADAM, weight_decay=WD_ADAMW)

def make_requires_grad(x: torch.Tensor) -> torch.Tensor:
    return x.detach().clone().requires_grad_(True)

x_min_per_sim_tr = df_train.groupby('__sim_id__')['dist_x'].transform('min').values
x_max_per_sim_tr = df_train.groupby('__sim_id__')['dist_x'].transform('max').values
is_left_tr  = (df_train['dist_x'].values == x_min_per_sim_tr)
is_right_tr = (df_train['dist_x'].values == x_max_per_sim_tr)
left_pos  = np.where(is_left_tr)[0]
right_pos = np.where(is_right_tr)[0]

X_inlet_t  = torch.tensor(X_train[left_pos],  dtype=torch.float32, device=device)                      
X_outlet_t = torch.tensor(X_train[right_pos], dtype=torch.float32, device=device).requires_grad_(True) 

Cref_inlet_t = Cref_train_t[left_pos]  


model.train()
num_train = X_train_t.shape[0]
steps_per_epoch = max(1, (num_train + BATCH_SIZE - 1) // BATCH_SIZE)

for epoch in range(EPOCHS_ADAM):
    perm = torch.randperm(num_train, device=device)
    total_loss_epoch = 0.0

    ramp = min(epoch / max(1, warmup_epochs), 1.0)
    lambda_phys  = lambda_phys_base  + (lambda_phys_max  - lambda_phys_base)  * ramp
    lambda_redox = lambda_redox_base + (lambda_redox_max - lambda_redox_base) * ramp

    for b in range(steps_per_epoch):
        idx = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]

        xb = make_requires_grad(X_train_t[idx])   
        yb = y_train_t[idx]

        optimizer.zero_grad()

        preds_scaled = model(xb)
        data_loss = nn.MSELoss()(preds_scaled, yb)

        pde_loss_sup = pde_residual_mean_abs(
            C_scaled=preds_scaled,
            xb_scaled=xb,
            mu_X_t=mu_X_t, std_X_t=std_X_t,
            mu_y_t=mu_y_t, std_y_t=std_y_t,
            device=device
        )

        if USE_COLLOCATION:
            colloc_idx = torch.randint(0, num_train, (idx.shape[0],), device=device)
            x_colloc = make_requires_grad(X_train_t[colloc_idx])
            preds_colloc = model(x_colloc)
            pde_loss_colloc = pde_residual_mean_abs(
                C_scaled=preds_colloc,
                xb_scaled=x_colloc,
                mu_X_t=mu_X_t, std_X_t=std_X_t,
                mu_y_t=mu_y_t, std_y_t=std_y_t,
                device=device
            )
        else:
            pde_loss_colloc = torch.tensor(0.0, device=device)

        preds_phys_np = (preds_scaled.detach().cpu().numpy() * scaler_y.scale_ + scaler_y.mean_).reshape(-1)
        df_batch = df_train.iloc[idx.cpu().numpy()].copy()
        df_batch['N(5)(mol/kgw)_predicted'] = np.clip(preds_phys_np, 0, None)

        nonneg_loss = torch.tensor(
            compute_nonnegativity_penalty(preds_phys_np),
            dtype=torch.float32, device=device
        )
        redox_loss  = torch.tensor(
            compute_redox_sequence_penalty(df_batch),
            dtype=torch.float32, device=device
        )

        pde_total = pde_loss_sup + COLLOC_MULTIPLIER * pde_loss_colloc
        loss = data_loss + lambda_phys * pde_total + lambda_nonneg * nonneg_loss + lambda_redox * redox_loss

        loss.backward()
        optimizer.step()
        total_loss_epoch += float(loss.item())

    X_outlet_t.requires_grad_(True)
    bc_in, bc_out = bc_losses(model, X_inlet_t, X_outlet_t, Cref_inlet_t, mu_X_t, std_X_t, mu_y_t, std_y_t)
    optimizer.zero_grad(set_to_none=True)
    bc_total = lambda_bc_in * bc_in + lambda_bc_out * bc_out
    bc_total.backward()
    optimizer.step()

    if epoch % 100 == 0 or epoch == EPOCHS_ADAM - 1:
        print(f"[AdamW] Epoch {epoch:4d} | Loss={total_loss_epoch/steps_per_epoch:.5e} "
              f"| Data={data_loss.item():.5e} PDE(sup)={pde_loss_sup.item():.5e} PDE(colloc)={pde_loss_colloc.item():.5e} "
              f"NonNeg={nonneg_loss.item():.5e} Redox={redox_loss.item():.5e} "
              f"| BC_in={bc_in.item():.5e} BC_out={bc_out.item():.5e} "
              f"| λ_phys={lambda_phys:.2f} λ_redox={lambda_redox:.2f}")


LBFGS_SUPERVISED_POINTS = 65536     
LBFGS_BC_POINTS         = 8192      
LBFGS_INCLUDE_PDE       = True

X_full_all = X_train_t
y_full_all = y_train_t

optimizer_lbfgs = optim.LBFGS(
    model.parameters(),
    max_iter=LBFGS_MAX_ITER,
    history_size=50,
    line_search_fn="strong_wolfe"
)

def closure():
    optimizer_lbfgs.zero_grad()

    if LBFGS_SUPERVISED_POINTS is None:
        idx_sup = torch.arange(X_full_all.shape[0], device=device)
    else:
        n_sup = min(LBFGS_SUPERVISED_POINTS, X_full_all.shape[0])
        idx_sup = torch.randint(0, X_full_all.shape[0], (n_sup,), device=device)

    Xb = X_full_all[idx_sup].detach().clone().requires_grad_(True)
    yb = y_full_all[idx_sup]
    preds_scaled_fb = model(Xb)
    data_loss_fb = nn.MSELoss()(preds_scaled_fb, yb)

    if LBFGS_INCLUDE_PDE:
        pde_fb = pde_residual_mean_abs(
            C_scaled=preds_scaled_fb,
            xb_scaled=Xb,
            mu_X_t=mu_X_t, std_X_t=std_X_t,
            mu_y_t=mu_y_t, std_y_t=std_y_t,
            device=device
        )
    else:
        pde_fb = torch.tensor(0.0, dtype=torch.float32, device=device)

    nonneg_loss_fb = torch.tensor(0.0, dtype=torch.float32, device=device)
    redox_loss_fb  = torch.tensor(0.0, dtype=torch.float32, device=device)

    if LBFGS_BC_POINTS is not None and LBFGS_BC_POINTS > 0:
        n_bc_in  = min(LBFGS_BC_POINTS, X_inlet_t.shape[0])
        n_bc_out = min(LBFGS_BC_POINTS, X_outlet_t.shape[0])

        sel_in  = torch.randint(0, X_inlet_t.shape[0],  (n_bc_in,),  device=device)
        sel_out = torch.randint(0, X_outlet_t.shape[0], (n_bc_out,), device=device)

        X_inlet_sub  = X_inlet_t[sel_in]                      
        Cref_in_sub  = Cref_inlet_t[sel_in]
        X_outlet_sub = X_outlet_t[sel_out].detach().clone().requires_grad_(True)  

        bc_in_fb, bc_out_fb = bc_losses(
            model, X_inlet_sub, X_outlet_sub, Cref_in_sub,
            mu_X_t, std_X_t, mu_y_t, std_y_t
        )
    else:
        bc_in_fb  = torch.tensor(0.0, dtype=torch.float32, device=device)
        bc_out_fb = torch.tensor(0.0, dtype=torch.float32, device=device)

    total = (
        data_loss_fb
        + lambda_phys_max * pde_fb
        + lambda_nonneg * nonneg_loss_fb
        + lambda_redox_max * redox_loss_fb
        + lambda_bc_in * bc_in_fb
        + lambda_bc_out * bc_out_fb
    )
    total.backward()
    return total

print("\n[LBFGS] Starting fine-tuning...")
optimizer_lbfgs.step(closure)


model.eval()

with torch.no_grad():
    X_test_pred = torch.tensor(X_test, dtype=torch.float32, device=device)
    pred_test_scaled = model(X_test_pred)
    pred_test_phys = (pred_test_scaled * std_y_t + mu_y_t).squeeze(1).cpu().numpy()

df_test_out = df_test.copy()
df_test_out['PINN_predicted'] = pred_test_phys

rmse = np.sqrt(mean_squared_error(df_test_out[output_column].values, df_test_out['PINN_predicted'].values))
mae  = mean_absolute_error(df_test_out[output_column].values, df_test_out['PINN_predicted'].values)
r2   = r2_score(df_test_out[output_column].values, df_test_out['PINN_predicted'].values)

X_test_pde = torch.tensor(X_test, dtype=torch.float32, device=device, requires_grad=True)
preds_test_scaled_t = model(X_test_pde)
pde_loss_test = pde_residual_mean_abs(
    C_scaled=preds_test_scaled_t,
    xb_scaled=X_test_pde,
    mu_X_t=mu_X_t, std_X_t=std_X_t,
    mu_y_t=mu_y_t, std_y_t=std_y_t,
    device=device
).item()

df_temp_test = df_test_out.copy()
df_temp_test['N(5)(mol/kgw)_predicted'] = np.clip(df_temp_test['PINN_predicted'].values, 0, None)
nonneg_loss_test = compute_nonnegativity_penalty(df_temp_test['N(5)(mol/kgw)_predicted'].values)
redox_loss_test  = compute_redox_sequence_penalty(df_temp_test)

print("\nFinal Evaluation on TEST (simulation-wise split):")
print(f"RMSE: {rmse:.6f}")
print(f"MAE : {mae:.6f}")
print(f"R²  : {r2:.6f}")
print("\nFinal Physical Losses (TEST):")
print(f"PDE Loss       : {pde_loss_test:.6e}")
print(f"Non-negativity : {nonneg_loss_test:.6e}")
print(f"Redox Penalty  : {redox_loss_test:.6e}")


stem = os.path.splitext(os.path.basename(DATA_PATH))[0]
pred_csv = f"{stem}_{KINETICS_MODE}_with_pinn_predictions_test.csv"
model_pt = f"surrogate_pinn_model_{KINETICS_MODE}.pt"

df_test_out = df_test_out.sort_values(['time', 'dist_x'])
df_test_out.to_csv("Benchmark2_predictions_test.csv", index=False)
torch.save(model.state_dict(), "surrogate_pinn.pt")

print("\nSaved files:")
print(" - Benchmark2_predictions_test.csv")
print(" - surrogate_pinn.pt")
